#### ⏰ Deadline: Wednesday, June 17, 20:00
### What is this notebook about? 

You may have heard about the COMPAS controversy, a famous debate in algorithmic fairness. The controversy focused on whether an algorithm that predicts recidivism (reoffending after a previous conviction) treated different racial groups fairly. As you probably have guessed - it did not. 

In this notebook, you will work with a simplified version of the COMPAS dataset and explore this controversy firsthand. Rather than focusing on formal definitions, you will investigate the underlying ideas intuitively. In the tutorial session itself, these insights will be contextualized within the broader landscape of fairness metrics.

🚀 Time to get started! 

---
#### 🟢 Station 1 - Assessing the model performance 

We will start with a simple model predicting recidivism rate from race, sex, age number of prior offenses (priors_count) and charge severity (c_charge_degree, felony vs. misdemeanor). The model is already implemented below to make it easier to get started. 

- Download the **Compass data** from the Jupyter Book under Data for Chapter 3
- Once you have the data, run the next code cells to get started!
- If you encounter any dependency errors, adjust the code (with the help of an LLM)!

📌 Note that the code has been generated with the help of an LLM but it has been inspected for the purposes of this exercise. Additional comments have been added to aid your unterstanding! 


In [ ]:
# Load the cleaned COMPAS dataset by specifying the path to the file 
# df = pd.read_csv("FILEPATH")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Keep only the two racial groups discussed in the COMPAS controversy
df = df[df["race"].isin(["African-American", "Caucasian"])].copy()

# Target: 1 = reoffended within two years
y = df["two_year_recid"].astype(int)

# Protected attribute used later for fairness analysis
race = df["race"]

# the train/test split is random
train_idx, test_idx = train_test_split(df.index,test_size=0.25,random_state=42)
base_features = ["sex", "age", "priors_count", "c_charge_degree"]

# Compare a model that can use race with one that cannot
model_specs = {"WITH race": base_features + ["race"],
    "WITHOUT race": base_features,}

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
trained_models = {}

for ax, (title, features) in zip(axes, model_specs.items()):
    X = df[features]

    # Separate categorical and numerical features
    cat_cols = X.select_dtypes(include="object").columns.tolist()
    num_cols = [c for c in features if c not in cat_cols]

    model = Pipeline(
        [("prep", ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols)])),
        ("clf", LogisticRegression(max_iter=1000))] 
    )

    # Train model
    model.fit(X.loc[train_idx], y.loc[train_idx])

    # True outcomes and race information for the test set
    # (used later to evaluate fairness across groups)
    
    y_test = y.loc[test_idx]
    race_test = race.loc[test_idx]

    # Predicted probability of recidivism
    y_score = model.predict_proba(X.loc[test_idx])[:, 1]

    # Convert probabilities into binary predictions
    y_pred = (y_score >= 0.5).astype(int)

    trained_models[title] = {
        "model": model,
        "features": features,
        "y_test": y_test,
        "race_test": race_test,
        "y_score": y_score,
        "y_pred": y_pred,}

    # Overall ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_score)
    ax.plot(fpr, tpr, linewidth=3, label=f"Overall AUC = {auc(fpr, tpr):.3f}")

    # ROC curves for each racial group
    for group in sorted(race_test.unique()):
        mask = race_test == group
        fpr_g, tpr_g, _ = roc_curve(y_test[mask], y_score[mask])
        ax.plot(fpr_g, tpr_g, label=f"{group} AUC = {auc(fpr_g, tpr_g):.3f}")

    # Random guessing baseline
    ax.plot([0, 1], [0, 1], "--", label="Random")

    ax.set_title(f"ROC: {title}")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend()

plt.tight_layout()
plt.show()

### 💡 Station 1 - Reflection
- Based on the results so far, what do you conclude regarding model fairness?
- Do you have enough information as of now? 
- What further information would you like to know, if any?

---
#### 🟢🟢 Station 2 - Fairness audit

There are many competing definitions and metrics of fairness. While they emphasize different notions of what it means to be fair, many commonly used metrics rely on information from the binary confusion matrix, such as illustrated below for our scenario. 

- What does each cell mean in terms of real-life consequences? 
- What would *your* fairness evaluation focus on? Why? Think of what real-life outcome/consequence you would like to minimize or maximize.

|                              | **Predicted: Will Reoffend** | **Predicted: Will Not Reoffend** |
|------------------------------|------------------------------|----------------------------------|
| **Actually Reoffends**       | True Positive (TP)           | False Negative (FN)             |
| **Actually Does Not Reoffend** | False Positive (FP)        | True Negative (TN)              |




---
#### 🟢🟢 Station 2 - Fairness audit - continued

- Time to perform the audit! Using the confusion matrix as your guide (remember each cell points to different real-life consequences!), compare model decisions for the two racial groups. You can illustrate differences in any table or plot form that you like!
  
**For simplicity, focus on the last model from the for loop above, i.e. the model without race information. It is already pre-loaded for you in the next cell.**

In [ ]:
# Your code here 

audit = model

### 💡 Station 2 - Reflection - What is your takeaway? 

---
#### 🟢🟢🟢 Station 3 - Final verdict - how fair is our model?

So far, we have focused on model performance and largely ignored the data itself. However, as you already know from the seminar and previous tutorial, historical and societal biases can be reflected in the data and carried over into the models trained on it.

A pragmatic argument may be that, regardless of the root cause, if one group exhibits a higher observed risk of recidivism, then a model should reflect that difference. After all, we haven't yet achieved a perfect world. 
Let's put this idea to test! 

- Check the recidivism rate for black and white populations **from the data alone**.
- Check the **predicted** recidivism rate of the model for the two populations.
- How do they compare? What do you conclude?


In [ ]:
# Your code here

### 💡 Station 3 - Reflection - What is your takeaway? How fair is the model? 

---
####  Can the tutor add your visualizations to the class notes? A gallery of student results helps see how different people approach the problem.

---
[Optional] 🤔💭 Was there something you struggled with? Are there any open questions?

Youe text here - 